In [ ]:
import os, re, ast, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from copy import deepcopy
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from torch.optim.swa_utils import AveragedModel, update_bn
from torch.utils.data import DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)

warnings.filterwarnings("ignore")
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

# ============================================================
# 1. LOAD DATA
# ============================================================
Label_data      = pd.read_excel("/content/train_fixed.xlsx")
unlabeled_data  = pd.read_excel("/content/unlabeled_fixed.xlsx")
validation_data = pd.read_excel("/content/DeepX_validation.xlsx")

# ============================================================
# 2. TEXT CLEANING
# ============================================================
def clean_arabic(text):
    if pd.isna(text):
        return ""
    text = str(text)
    text = re.sub(r"[إأآا]", "ا", text)
    text = re.sub(r"ى","ي", text)
    text = re.sub(r"ة","ه", text)
    text = re.sub(r"[^\w\s]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

for df in [Label_data, unlabeled_data, validation_data]:
    df["review_text"] = df["review_text"].apply(clean_arabic)

# ============================================================
# 3. CATEGORY MAPPING
# ============================================================
category_mapping = {
    "مطعم": "restaurant", "مطعم مأكولات ومشروبات": "restaurant",
    "مطعم يمني": "restaurant", "مطعم أطباق اللحوم": "restaurant",
    "مطعم مأكولات الشرق الأوسط": "restaurant", "مطعم بيتزا": "restaurant",
    "مطعم فلافل": "restaurant", "مطعم مأكولات مشوية": "restaurant",
    "مطعم مأكولات لبنانية": "restaurant", "مطعم مأكولات سورية": "restaurant",
    "مطعم مأكولات تركية": "restaurant", "مطعم دجاج": "restaurant",
    "مطعم مأكولات إيطالية": "restaurant", "مطعم مأكولات أمريكية": "restaurant",
    "مطعم شاورما": "restaurant", "مطعم مأكولات هندية": "restaurant",
    "مطعم وجبات سريعة": "restaurant", "مطعم للإفطار": "restaurant",
    "مطعم ببوفيه": "restaurant", "مطعم مأكولات مصرية": "restaurant",
    "مطعم عائلي": "restaurant", "مطعم مأكولات فرنسية": "restaurant",
    "مطعم مأكولات بحرية": "restaurant", "مطعم كريب": "restaurant",
    "مطعم مأكولات من جنوب إفريقيا": "restaurant",
    "كافيه": "cafe", "مقهى": "cafe",
    "عِيادة أسنان": "medical", "صيدلية": "medical", "مركز طبي": "medical",
    "مستشفى متخصص": "medical", "طبيب أسنان": "medical", "مستشفى": "medical",
    "عيادة طبية": "medical", "مستشفى جامعي": "medical", "المستشفى العسكري": "medical",
    "عيادة الخصوبة": "medical", "عيادة متخصصة": "medical", "مستشفى عام": "medical",
    "المستشفى الحكومي": "medical", "مستشفى خاصة": "medical", "عيادة جراحة التجميل": "medical",
    "صالة رياضة": "gym", "غرفة لياقة": "gym", "برنامج اللياقة البدنية": "fitness_program",
    "صالون تجميل": "salon", "صالون عناية بالأظافر": "salon",
    "مصفف الشعر": "barber", "صالون حلاقة": "barber",
    "فندق": "hotel",
    "متجر": "retail", "منفذ بيع بالتجزئة": "retail", "سوق": "retail",
    "مركز تسوق": "retail", "متجر بقالة": "supermarket", "سوبرماركت": "supermarket",
    "متجر بقالة راقٍ": "supermarket",
    "متجر ملابس حريمي": "clothing_store", "متجر ملابس": "clothing_store",
    "متجر ملابس رجالي": "clothing_store", "متجر ملابس رياضية": "clothing_store",
    "ecommerce": "ecommerce", "travel": "travel", "transport": "transport",
    "food_delivery": "food_delivery", "real_estate": "real_estate",
    "entertainment": "entertainment",
}

def add_domain(df):
    df = df.copy()
    df["business_category"] = df["business_category"].astype(str).str.strip().str.lower()
    df["business_domain"]   = df["business_category"].map(category_mapping).fillna("unknown")
    df["text_with_domain"]  = "[domain: " + df["business_domain"] + "] " + df["review_text"]
    return df

Label_data      = add_domain(Label_data)
unlabeled_data  = add_domain(unlabeled_data)
validation_data = add_domain(validation_data)

# ============================================================
# 4. SAFE PARSER + EXPAND LABELED ROWS
# ============================================================
def safe_eval(val):
    if pd.isna(val) if not isinstance(val, (list, dict)) else False:
        return None
    if isinstance(val, (list, dict)):
        return val
    try:
        return ast.literal_eval(str(val))
    except Exception:
        return None

Label_data["aspects"]          = Label_data["aspects"].apply(safe_eval)
Label_data["aspect_sentiments"]= Label_data["aspect_sentiments"].apply(safe_eval)
Label_data = Label_data.dropna(subset=["aspects","aspect_sentiments"]).reset_index(drop=True)
Label_data["aspects"] = Label_data["aspects"].apply(lambda x: [a.lower().strip() for a in x])

rows_label = []
for _, row in Label_data.iterrows():
    if not isinstance(row["aspect_sentiments"], dict):
        continue
    for aspect, sentiment in row["aspect_sentiments"].items():
        rows_label.append({
            "text"    : row["text_with_domain"],
            "aspect"  : aspect.lower().strip(),
            "platform": row["platform"],
            "label"   : sentiment,
        })

labeled_df = pd.DataFrame(rows_label)
print(f"Labeled rows: {len(labeled_df)}")

# ============================================================
# 5. ENCODE LABELS  (column MUST be 'labels' for HF Trainer)
# ============================================================
le = LabelEncoder()
labeled_df["labels"] = le.fit_transform(labeled_df["label"])
labeled_df = labeled_df.drop(columns=["label"])
label_classes = le.classes_
NUM_LABELS    = len(label_classes)
print(f"Classes ({NUM_LABELS}): {label_classes}")

# ============================================================
# 6. FINAL TEXT
# ============================================================
def make_final_text(df):
    return (df["text"].astype(str)
            + " [ASPECT] " + df["aspect"].astype(str)
            + " [PLATFORM] " + df["platform"].astype(str))

labeled_df["final_text"] = make_final_text(labeled_df)

# ============================================================
# 7. UNLABELED EXPANSION
# ============================================================
all_aspects = labeled_df["aspect"].unique().tolist()
print(f"Unique aspects: {len(all_aspects)}")

rows_unlabeled = []
for _, row in unlabeled_data.iterrows():
    for aspect in all_aspects:
        rows_unlabeled.append({
            "text"    : row["text_with_domain"],
            "aspect"  : aspect,
            "platform": row["platform"],
        })
unlabeled_expanded = pd.DataFrame(rows_unlabeled)
unlabeled_expanded["final_text"] = make_final_text(unlabeled_expanded)
print(f"Unlabeled expanded: {unlabeled_expanded.shape}")

# ============================================================
# 8. VALIDATION DATA
# ============================================================
rows_val = []
for _, row in validation_data.iterrows():
    aspect_dict = safe_eval(row.get("aspect_sentiments", None))
    if not isinstance(aspect_dict, dict):
        continue
    for aspect, sentiment in aspect_dict.items():
        rows_val.append({
            "text"    : row["text_with_domain"],
            "aspect"  : aspect.lower().strip(),
            "platform": row["platform"],
            "label"   : sentiment,
        })

val_df = pd.DataFrame(rows_val)
if not val_df.empty:
    val_df = val_df[val_df["label"].isin(le.classes_)].reset_index(drop=True)
    val_df["labels"]     = le.transform(val_df["label"])
    val_df["final_text"] = make_final_text(val_df)
    print(f"Validation rows: {len(val_df)}")
else:
    val_df = None
    print("Warning: no validation aspect_sentiments found.")

# ============================================================
# 9. FOCAL LOSS
# ============================================================
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None):
        super().__init__()
        self.gamma  = gamma
        self.weight = weight

    def forward(self, logits, targets):
        ce   = F.cross_entropy(logits, targets, weight=self.weight, reduction="none")
        pt   = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()

# ============================================================
# 10. EMA  (Exponential Moving Average of weights)
# ============================================================
class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for k, v in model.named_parameters():
            self.shadow[k] = v.detach().clone().float().to(v.device)

    def update(self, model):
        for k, v in model.named_parameters():
            v_data = v.detach().float()
            self.shadow[k] = self.shadow[k].to(v.device)
            self.shadow[k] = self.decay * self.shadow[k] + (1 - self.decay) * v_data

    def apply_shadow(self, model):
        for k, v in model.named_parameters():
            self.backup[k] = v.data.clone()
            v.data.copy_(self.shadow[k].to(v.device))

    def restore(self, model):
        for k, v in model.named_parameters():
            v.data.copy_(self.backup[k].to(v.device))

# ============================================================
# 11. FGM  (Fast Gradient Method adversarial training)
# ============================================================
class FGM:
    def __init__(self, model, epsilon=0.5, emb_name="word_embeddings"):
        self.model    = model
        self.epsilon  = epsilon
        self.emb_name = emb_name
        self.backup   = {}

    def attack(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name:
                self.backup[name] = param.data.clone()
                norm = torch.norm(param.grad)
                if norm != 0:
                    r_at = self.epsilon * param.grad / norm
                    param.data.add_(r_at)

    def restore(self):
        for name, param in self.model.named_parameters():
            if param.requires_grad and self.emb_name in name:
                param.data = self.backup[name]
        self.backup = {}

# ============================================================
# 12. CUSTOM TRAINER (Focal + R-Drop + FGM + EMA) - FIXED SIGNATURE
# ============================================================
class AdvancedTrainer(Trainer):
    def __init__(self, *args, ema=None, fgm=None,
                 focal_gamma=2.0, rdrop_alpha=4.0, **kwargs):
        super().__init__(*args, **kwargs)
        self.ema         = ema
        self.fgm         = fgm
        self.focal_loss  = FocalLoss(gamma=focal_gamma)
        self.rdrop_alpha = rdrop_alpha

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None, **kwargs):
        labels = inputs.pop("labels")
        
        if not model.training:
            outputs = model(**inputs)
            if hasattr(outputs, 'loss') and outputs.loss is not None:
                loss = outputs.loss
            else:
                loss = self.focal_loss(outputs.logits, labels)
            return (loss, outputs) if return_outputs else loss

        out1 = model(**inputs)
        loss1 = self.focal_loss(out1.logits, labels)

        out2 = model(**inputs)
        loss2 = self.focal_loss(out2.logits, labels)
        p1 = F.log_softmax(out1.logits, dim=-1)
        p2 = F.log_softmax(out2.logits, dim=-1)
        kl = (F.kl_div(p1, p2.exp(), reduction="batchmean") +
              F.kl_div(p2, p1.exp(), reduction="batchmean")) / 2
        loss = (loss1 + loss2) / 2 + self.rdrop_alpha * kl

        if self.fgm is not None:
            loss.backward(retain_graph=True)
            self.fgm.attack()
            out_adv = model(**inputs)
            loss_adv = self.focal_loss(out_adv.logits, labels)
            loss_adv.backward()
            self.fgm.restore()

        if self.ema is not None:
            self.ema.update(model)

        return (loss, out1) if return_outputs else loss

# ============================================================
# 13. CROSS-VALIDATION + PSEUDO-LABEL LOOP
# ============================================================
models_to_train = [
    "aubmindlab/bert-base-arabertv2",
    "UBC-NLP/MARBERT",
]

skf             = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
oof_preds       = np.zeros((len(labeled_df), NUM_LABELS))
test_preds_all  = []      # one entry per model

for model_name in models_to_train:
    print(f"\n{'='*60}\nMODEL: {model_name}\n{'='*60}")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    collator  = DataCollatorWithPadding(tokenizer)

    def tok_fn(batch):
        return tokenizer(batch["final_text"],
                         truncation=True, padding=False, max_length=256)

    labelled_ds  = Dataset.from_pandas(labeled_df[["final_text","labels"]])
    labelled_ds  = labelled_ds.map(tok_fn, batched=True)

    unlabeled_ds = Dataset.from_pandas(unlabeled_expanded[["final_text"]])
    unlabeled_ds = unlabeled_ds.map(tok_fn, batched=True)

    model_unlab_preds = np.zeros((len(unlabeled_expanded), NUM_LABELS))

    for fold, (tr_idx, val_idx) in enumerate(
            skf.split(labeled_df, labeled_df["labels"])):
        print(f"\n── Fold {fold} ──")

        model = AutoModelForSequenceClassification.from_pretrained(
            model_name, num_labels=NUM_LABELS)

        ema = EMA(model, decay=0.999)
        fgm = FGM(model, epsilon=0.5)

        # Compute warmup steps (10% of total training steps)
        train_dataset_len = len(tr_idx)
        total_steps = train_dataset_len // 8 * 3  # 3 epochs, batch size 8
        warmup_steps = int(0.1 * total_steps)

        args = TrainingArguments(
            output_dir            = f"./{model_name.split('/')[-1]}_fold{fold}",
            learning_rate         = 2e-5,
            num_train_epochs      = 3,
            per_device_train_batch_size = 8,
            per_device_eval_batch_size  = 16,
            save_strategy         = "no",
            fp16                  = torch.cuda.is_available(),
            report_to             = "none",
            dataloader_drop_last  = False,
            warmup_steps          = warmup_steps,
            weight_decay          = 0.01,
        )

        trainer = AdvancedTrainer(
            model         = model,
            args          = args,
            train_dataset = labelled_ds.select(tr_idx),
            eval_dataset  = labelled_ds.select(val_idx),
            data_collator = collator,
            ema           = ema,
            fgm           = fgm,
            focal_gamma   = 2.0,
            rdrop_alpha   = 4.0,
        )

        trainer.train()

        # apply EMA weights for inference
        ema.apply_shadow(model)

        # OOF predictions
        val_logits  = trainer.predict(labelled_ds.select(val_idx)).predictions
        val_probs   = torch.softmax(torch.tensor(val_logits), dim=1).numpy()
        oof_preds[val_idx] += val_probs

        # Unlabeled predictions
        test_logits = trainer.predict(unlabeled_ds).predictions
        test_probs  = torch.softmax(torch.tensor(test_logits), dim=1).numpy()
        model_unlab_preds += test_probs / skf.n_splits

        # restore original weights after inference
        ema.restore(model)

    test_preds_all.append(model_unlab_preds)

# ============================================================
# 14. OOF SCORE
# ============================================================
oof_f1 = f1_score(labeled_df["labels"],
                  oof_preds.argmax(axis=1), average="macro")
print(f"\n✅ OOF Macro F1: {oof_f1:.4f}")

# ============================================================
# 15. ENSEMBLE UNLABELED PREDICTIONS
# ============================================================
ensemble_probs = np.mean(test_preds_all, axis=0)

# ============================================================
# 16. ENTROPY-BASED PSEUDO-LABEL SELECTION
# ============================================================
def entropy(probs):
    eps  = 1e-9
    return -(probs * np.log(probs + eps)).sum(axis=1)

ent              = entropy(ensemble_probs)
conf             = ensemble_probs.max(axis=1)
pseudo_labels    = ensemble_probs.argmax(axis=1)

MAX_ENTROPY      = -np.log(1/NUM_LABELS) * 0.25   # 25% of max possible entropy
CONF_THRESHOLD   = 0.90

mask = (conf > CONF_THRESHOLD) & (ent < MAX_ENTROPY)
pseudo_df = unlabeled_expanded[mask].copy()
pseudo_df["labels"] = pseudo_labels[mask]

print(f"\nPseudo-labeled samples selected: {len(pseudo_df)}")
print(f"  (confidence>{CONF_THRESHOLD} AND entropy<{MAX_ENTROPY:.3f})")

# ============================================================
# 17. FINAL TRAINING SET
# ============================================================
final_train_df = pd.concat([
    labeled_df[["final_text","labels"]],
    pseudo_df[["final_text","labels"]],
], ignore_index=True)
print(f"Final training set: {len(final_train_df)} rows")
final_train_df.to_csv("final_training_data.csv", index=False)

# ============================================================
# 18. FINAL MODEL TRAINING (AraBERT + SWA)
# ============================================================
print("\n" + "="*60)
print("TRAINING FINAL MODEL WITH SWA")
print("="*60)

FINAL_MODEL_NAME = "aubmindlab/bert-base-arabertv2"
tokenizer_f = AutoTokenizer.from_pretrained(FINAL_MODEL_NAME)
collator_f  = DataCollatorWithPadding(tokenizer_f)

def tok_final(batch):
    return tokenizer_f(batch["final_text"],
                       truncation=True, padding=False, max_length=256)

final_ds = Dataset.from_pandas(final_train_df[["final_text","labels"]])
final_ds = final_ds.map(tok_final, batched=True)

if val_df is not None:
    val_ds = Dataset.from_pandas(val_df[["final_text","labels"]])
    val_ds = val_ds.map(tok_final, batched=True)
else:
    val_ds = None

base_model = AutoModelForSequenceClassification.from_pretrained(
    FINAL_MODEL_NAME, num_labels=NUM_LABELS).to(device)

swa_model = AveragedModel(base_model)
ema_final = EMA(base_model, decay=0.999)
fgm_final = FGM(base_model, epsilon=0.5)

total_steps_final = (len(final_ds) // 8) * 5   # 5 epochs, batch 8
warmup_steps_final = int(0.1 * total_steps_final)

final_args = TrainingArguments(
    output_dir                  = "./final_model",
    learning_rate               = 2e-5,
    num_train_epochs            = 5,
    per_device_train_batch_size = 8,
    per_device_eval_batch_size  = 16,
    save_strategy               = "epoch",
    evaluation_strategy         = "epoch" if val_ds else "no",
    load_best_model_at_end      = val_ds is not None,
    metric_for_best_model       = "eval_loss",
    fp16                        = torch.cuda.is_available(),
    report_to                   = "none",
    warmup_steps                = warmup_steps_final,
    weight_decay                = 0.01,
)

final_trainer = AdvancedTrainer(
    model         = base_model,
    args          = final_args,
    train_dataset = final_ds,
    eval_dataset  = val_ds,
    data_collator = collator_f,
    ema           = ema_final,
    fgm           = fgm_final,
    focal_gamma   = 2.0,
    rdrop_alpha   = 4.0,
)

final_trainer.train()

# SWA: update batch norm statistics
print("\nApplying SWA averaging …")
swa_model.update_parameters(base_model)

def collate_for_bn(batch):
    return collator_f(batch)

bn_loader = DataLoader(
    final_ds.with_format("torch",
        columns=["input_ids","attention_mask","token_type_ids","labels"],
        output_all_columns=False),
    batch_size=16,
    collate_fn=collate_for_bn,
)
update_bn(bn_loader, swa_model, device=device)

# Apply EMA on top of SWA for inference
ema_final.apply_shadow(base_model)

# ============================================================
# 19. FINAL EVALUATION
# ============================================================
if val_ds is not None:
    val_preds      = final_trainer.predict(val_ds)
    val_probs      = torch.softmax(torch.tensor(val_preds.predictions), dim=1).numpy()
    val_pred_cls   = val_probs.argmax(axis=1)
    val_f1         = f1_score(val_df["labels"], val_pred_cls, average="macro")
    print(f"\n🏆 Final Validation Macro F1: {val_f1:.4f}")

# ============================================================
# 20. SAVE
# ============================================================
final_trainer.save_model("./final_model")
tokenizer_f.save_pretrained("./final_model")
print("\nModel saved to ./final_model")
print("DONE ✅")